### Create Training data for binary classification: description of business sector - True False

In [98]:
import pandas as pd
import glob
import os
import ast
import json

import numpy as np
import pandas as pd
import os
import sys
import glob
import tqdm
import re
import random
from typing import List
from sklearn.model_selection import train_test_split

sys.path.append("../../")
from sentence_splitter import split_text_into_sentences
#from BERT_classifier.Classify_report_with_BERT import classification_report_BERT
#from transformers import AutoModelForSequenceClassification, AutoTokenizer

#### Load Positives

In [10]:
positives_json_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/datasets/stoxx_600/JSONs"

In [11]:
df_overview = pd.read_csv("/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/datasets/stoxx_600/stoxx_600_overview.csv", sep=";")
df_positives = df_overview[df_overview["description_page"].notna()]
df_positives

,Unnamed: 0,Name,Symbol,FactSet ID,Revenue - 2022 (in EUR),Revenue - 2023 (in EUR),Revenue - 2024 (in EUR),NACE,NACE_letter,Report,description_page
3,107,AAK AB,AAK-SE,AAK-SE,4741.086317,4010.433824,3939.34280627966,10.89,C,AAK AB1.pdf,3
6,94,ABB Ltd.,ABBN-CH,ABBN-CH,28073.948473,29665.651960,30586.3762461154,27.11,C,ABB Ltd.2.pdf,18
9,135,Accelleron Industries AG,ACLN-CH,ACLN-CH,742.680345,846.217532,NaN,28.11,C,Accelleron Industries AG1.pdf,7
10,296,Acciona SA,ANA-ES,ANA-ES,11195.000000,17021.000000,19190,41.20,F,Acciona SA2.pdf,7
11,365,Accor SA,AC-FR,AC-FR,4224.000000,5056.000000,5606,55.10,I,Accor SA1.pdf,5
...,...,...,...,...,...,...,...,...,...,...,...
394,69,Orkla ASA,ORK-NO,ORK-NO,5774.960695,5932.590483,6073.67720031738,10.89,C,Orkla ASA1.pdf,11
466,598,Scout24 SE,G24-DE,G24-DE,447.539000,509.114000,s,96.09,S,Scout24 SE3.pdf,41
472,286,Severn Trent Plc,SVT-GB,SVT-GB,2505.264229,2709.430915,NaN,36.00,E,Severn Trent Plc1.pdf,8
479,133,Siemens Energy AG,ENR-DE,ENR-DE,29005.000000,31119.000000,34465,27.33,C,Siemens Energy AG1.pdf,5


In [12]:
for i, row in df_positives.iterrows(): 
    path = row["Report"].replace(".pdf", ".json")
    with open(os.path.join(positives_json_path, path), "r") as f: 
        report_json = json.load(f)
    
    description_pages = ast.literal_eval(row["description_page"])
    description_pages = [description_pages] if isinstance(description_pages, int) else description_pages

    description_text = ""
    for description_page in description_pages:

        description_text += list(filter(lambda x: x["page"] == description_page + 1, report_json["pages"]))[0]["markdown"]

    print(row["Report"])
    print(description_text)
    print("-----" * 5 + "\n"*6)

    df_positives.loc[i, "Description"] = description_text

AAK AB1.pdf
## we do is about Making Better Happen ™ Everything

AAK specializes in plant-based oils and fats, the value-adding ingredients in many products people love to consume. We make these products better tasting, healthier, and more sustainable. In addition, we enhance their sensory experience - by giving the silkier mouthfeel in premium chocolate, the juicier texture in a plant-based burger, and a puffier appearance in a lower-fat pastry.

We can also optimize our customers' production and processes by substituting existing ingredients with plantbased equivalents that improve efficiency and enhance the performance and sustainability of the end product. AAK's value-adding solutions enable our customers to Making Better Happen ™ .

At the heart of AAK's offer is customer co-development, combining our desire to understand what Making Better Happen ™  means for each customer with the unique flexibility of our production assets and deep knowledge of products and industries, includin

/var/folders/fp/yhl61lbj3m73x3_17tsp1vrr0000gn/T/ipykernel_70545/2710329527.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_positives.loc[i, "Description"] = description_text


In [13]:
# # store page

# to_folder = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/datasets/stoxx_600/company_descriptions_txt"

# for i, row in df_positives.iterrows(): 
#     print(row["Description"])
#     print(row["Report"])
#     with open(to_folder + "/" + row["Report"].replace("pdf", "txt") , "w") as f: 
#         f.write(row["Description"])

In [14]:
def get_tables(lines: list): 
    tables = []
    current_table = []

    for line in lines:
        if line.strip().startswith("|"):  # line belongs to a table
            current_table.append(line.strip())
        else:
            if current_table:  # table ended
                tables.append("\n".join(current_table))
                current_table = []

    # catch last table if file ends without empty lines
    if current_table:
        tables.append("\n".join(current_table))

    return tables

def preprocess_report(text: str) -> List[str]:

    # with open(pdf_path, "r") as f: 
    #     text = f.read()

    sentence_length = 1
    
    lines = text.split("\n")

    tables = get_tables(lines)

    # drop if condidtion is True
    conditions = [
        # filter images
        #lambda line: line == '<!-- image -->',
        
        #filter tables 
        #lambda line: (line[0] == "|" and line[-1] == "|") if len(line) > 1 else False, 

        # filter headers
        lambda line: line.strip()[0] == "#" if len(line) > 0 else True,

        # filter sentences
        #lambda line: "." not in line,
        
        # more than 50% is numbers
        #lambda line: sum(ch.isalpha() for ch in line) / len(line) < 0.5,

        # minimum 3 words 
        #lambda line: len(re.sub(r"[^a-zA-ZäöüÄÖÜß\s]", '', line).strip().split(" ")) < 3,

        # Minimum 2 Sentences
        #lambda line: sum([0 if len(sentence.split(" ")) < 3 else 1 for sentence in split_text_into_sentences(line, "en")]) < 2

    ]
    accepted_lines = [line for line in lines if not any(condition(line) for condition in conditions)]
    accepted_lines += tables

    chunks = []

    for line in accepted_lines: 
        sentences = split_text_into_sentences(line, language='en')
        sentences = [sentence.strip() for sentence in sentences]
        sentences = [sentence for sentence in sentences if sentence != ""]
        new_chunks = [(" ".join(sentences[i:i+sentence_length])).strip() for i in range(0, len(sentences), 3)]

        chunks += new_chunks
    
    if len(chunks) <= 1: 
        return chunks

    # if there is only one sentence in the last chunk, balance the two last chunks
    if len(split_text_into_sentences(chunks[-1], language = "en")) == 1: 
        last_two_chunks = chunks[-2] + " " + chunks[-1]
        chunks[-2] = last_two_chunks[0:(len(last_two_chunks) + 1) // 2]
        chunks[-1] = last_two_chunks[(len(last_two_chunks) + 1) // 2: (len(last_two_chunks)) - (len(last_two_chunks) + 1) // 2]

    chunks = [re.sub(r'\b\d+\.\d+\b', '', chunk) for chunk in chunks]
    chunks = [re.sub(r"[^a-zA-ZäöüÄÖÜß.\s]", '', chunk) for chunk in chunks]
    chunks = [re.sub(r"\s+", " ", chunk) for chunk in chunks]
    chunks = [re.sub(r'\.{2,}', " ", chunk) for chunk in chunks]
    chunks = [re.sub(r'^\d+\.\s*', " ", chunk) for chunk in chunks]
    chunks = [chunk.lower() for chunk in chunks]
    chunks = [chunk.strip() for chunk in chunks]

    return chunks

In [15]:
df_positives.loc[:,"lenght_desc"] = df_positives["Description"].apply(len)
df_positives

/var/folders/fp/yhl61lbj3m73x3_17tsp1vrr0000gn/T/ipykernel_70545/3719149439.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_positives.loc[:,"lenght_desc"] = df_positives["Description"].apply(len)


,Unnamed: 0,Name,Symbol,FactSet ID,Revenue - 2022 (in EUR),Revenue - 2023 (in EUR),Revenue - 2024 (in EUR),NACE,NACE_letter,Report,description_page,Description,lenght_desc
3,107,AAK AB,AAK-SE,AAK-SE,4741.086317,4010.433824,3939.34280627966,10.89,C,AAK AB1.pdf,3,## we do is about Making Better Happen ™ Every...,1424
6,94,ABB Ltd.,ABBN-CH,ABBN-CH,28073.948473,29665.651960,30586.3762461154,27.11,C,ABB Ltd.2.pdf,18,-\n\n## Who we are\n\nABB has a history of inn...,2907
9,135,Accelleron Industries AG,ACLN-CH,ACLN-CH,742.680345,846.217532,NaN,28.11,C,Accelleron Industries AG1.pdf,7,## Accelleron at a glance\n\nAccelleron's tech...,1491
10,296,Acciona SA,ANA-ES,ANA-ES,11195.000000,17021.000000,19190,41.20,F,Acciona SA2.pdf,7,## 2. BUSINESS MODEL: BUSINESS AS UNUSUAL\n\nA...,1569
11,365,Accor SA,AC-FR,AC-FR,4224.000000,5056.000000,5606,55.10,I,Accor SA1.pdf,5,"## Message from Sébastien Bazin, Chairman and ...",3503
...,...,...,...,...,...,...,...,...,...,...,...,...,...
394,69,Orkla ASA,ORK-NO,ORK-NO,5774.960695,5932.590483,6073.67720031738,10.89,C,Orkla ASA1.pdf,11,"## Orkla's business areas in 2022\n\nIn 2022, ...",753
466,598,Scout24 SE,G24-DE,G24-DE,447.539000,509.114000,s,96.09,S,Scout24 SE3.pdf,41,≡\n\n## Grundlagen des Konzerns\n\n## Geschäft...,3264
472,286,Severn Trent Plc,SVT-GB,SVT-GB,2505.264229,2709.430915,NaN,36.00,E,Severn Trent Plc1.pdf,8,## OUR BUSINESS MODEL UNDERSTANDING OUR WORLD\...,2100
479,133,Siemens Energy AG,ENR-DE,ENR-DE,29005.000000,31119.000000,34465,27.33,C,Siemens Energy AG1.pdf,5,## Our business areas\n\n## Digital Industries...,4527


In [32]:
paragraphs = pd.DataFrame(columns=["text", "report"])
for i, row in df_positives.iterrows():
    df_temp = pd.DataFrame(data=preprocess_report(row["Description"]), columns=["text"])
    df_temp["report"] = row["Report"]
    paragraphs = pd.concat([paragraphs, df_temp], ignore_index=True)

In [42]:
paragraphs.drop_duplicates()
paragraphs = paragraphs[paragraphs["text"] != ""]
paragraphs = paragraphs[paragraphs["text"].apply(len) > 30]
paragraphs

,text,report
0,aak specializes in plantbased oils and fats th...,AAK AB1.pdf
1,we can also optimize our customers production ...,AAK AB1.pdf
2,at the heart of aaks offer is customer codevel...,AAK AB1.pdf
5,abb has a history of innovation excellence str...,ABB Ltd.2.pdf
6,our strong heritage as a technology pioneer ha...,ABB Ltd.2.pdf
...,...,...
1516,we are building a customerfocused organization...,Signify NV3.pdf
1517,we are developing tiered offerings with multip...,Signify NV3.pdf
1518,we are driving new sustainable growth areas to...,Signify NV3.pdf
1519,we are creating a digital front and backend em...,Signify NV3.pdf


In [41]:
# these have been selected by hand!
positive_paragraphs_selected = pd.read_csv("/Users/hendrikweichel/Desktop/positive_data_selected.csv", sep=";", index_col=0)
positive_paragraphs_selected = positive_paragraphs_selected.dropna()
positive_paragraphs_selected

,0
0,aak specializes in plantbased oils and fats th...
1,we can also optimize our customers production ...
2,at the heart of aaks offer is customer codevel...
3,abb has a history of innovation excellence str...
4,our strong heritage as a technology pioneer ha...
...,...
754,we are building a customerfocused organization...
755,we are developing tiered offerings with multip...
756,we are driving new sustainable growth areas to...
757,we are creating a digital front and backend em...


In [50]:
positive_paragraphs = pd.merge(paragraphs, positive_paragraphs_selected, left_on="text", right_on="0", how="inner")
positive_paragraphs = positive_paragraphs.drop(columns="0")
positive_paragraphs = positive_paragraphs.drop_duplicates(subset="text")

#### Load Negatives

In [65]:
negative_paragraphs = pd.DataFrame(columns=["text", "report"])

for i, row in df_positives.iterrows(): 
    path = row["Report"].replace(".pdf", ".json")
    with open(os.path.join(positives_json_path, path), "r") as f: 
        report_json = json.load(f)
    description_pages = ast.literal_eval(row["description_page"])
    description_pages = [description_pages] if isinstance(description_pages, int) else description_pages

    description_text = ""
    for description_page in description_pages:

        description_text += list(filter(lambda x: x["page"] != description_page + 1, report_json["pages"]))[np.random.randint(len(report_json["pages"])-1)]["markdown"]

    df_temp = pd.DataFrame(data=preprocess_report(description_text), columns=["text"])
    df_temp["report"] = row["Report"]

    print(row["Report"])
    print(description_text)
    print("-----" * 5 + "\n"*6)

    negative_paragraphs = pd.concat([negative_paragraphs, df_temp], ignore_index=True)

AAK AB1.pdf
Annual Report 2022

## The Multi-oil Ingredient House

<!-- image -->
-------------------------






ABB Ltd.2.pdf
Given the macro landscape, the Company continued its focus on solid growth fundamentals whilst reinforcing the highest ESG standards with resilience. ABB India remains vigilant on risks emanating from COVID19 and its variants with a focus on employee well-being.  The Company's agile business model and portfolio ensured conversion of opportunities, maximally optimizing  the  country's  conducive  and  relatively  stable environment in an otherwise volatile global weather, leading to a quantum leap in performance, back to pre-pandemic and preportfolio realignment levels. Cost efficiency programs together with strategic investments for new facilities, product launches, and launch of online business models supported this journey of profitable growth.

## Operational performance

Though the year started off with the Omicron scare, 2022 was the year that characteriz

In [80]:
negative_paragraphs.drop_duplicates()
negative_paragraphs = negative_paragraphs[negative_paragraphs["text"] != ""]
negative_paragraphs = negative_paragraphs[negative_paragraphs["text"].apply(len) > 30]
negative_paragraphs

,text,report
2,given the macro landscape the company continue...,ABB Ltd.2.pdf
3,cost efficiency programs together with strateg...,ABB Ltd.2.pdf
4,though the year started off with the omicron s...,ABB Ltd.2.pdf
5,during the year three milestone facilities wer...,ABB Ltd.2.pdf
6,abbs portfolio of sustainable technology solut...,ABB Ltd.2.pdf
...,...,...
1697,these are fully paid ordinary shares.,Signify NV3.pdf
1701,issued subscribed and paidup capital,Signify NV3.pdf
1705,shares fully paid for consideration other,Signify NV3.pdf
1712,less allowance for expected credit losses,Signify NV3.pdf


In [81]:
df_negative_paragraphs = negative_paragraphs.copy()

### Combine

In [82]:
df_negative_paragraphs["label"] = False
positive_paragraphs["label"] = True

In [83]:
df_negative_paragraphs.columns, positive_paragraphs.columns

(Index(['text', 'report', 'label'], dtype='object'),
 Index(['text', 'report', 'label'], dtype='object'))

In [84]:
df_negative_paragraphs = df_negative_paragraphs.rename(columns={0:"text"})
positive_paragraphs = positive_paragraphs.rename(columns={"0":"text"})

In [85]:
df_full = pd.concat((df_negative_paragraphs, positive_paragraphs), axis=0)

In [86]:
df_full = df_full.reset_index(drop=True)
df_full

,text,report,label
0,given the macro landscape the company continue...,ABB Ltd.2.pdf,False
1,cost efficiency programs together with strateg...,ABB Ltd.2.pdf,False
2,though the year started off with the omicron s...,ABB Ltd.2.pdf,False
3,during the year three milestone facilities wer...,ABB Ltd.2.pdf,False
4,abbs portfolio of sustainable technology solut...,ABB Ltd.2.pdf,False
...,...,...,...
1500,we are building a customerfocused organization...,Signify NV3.pdf,True
1501,we are developing tiered offerings with multip...,Signify NV3.pdf,True
1502,we are driving new sustainable growth areas to...,Signify NV3.pdf,True
1503,we are creating a digital front and backend em...,Signify NV3.pdf,True


In [109]:
#df_full.to_csv("/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/training_data/company_description_2/full_data.csv")
df_full = pd.read_csv("/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/training_data/company_description_2/full_data.csv", sep=";")

### Split

In [110]:
df_full
reports = df_full["report"].unique()

In [111]:
train_reports, temp_reports = train_test_split(
    reports,
    test_size=0.40,
    random_state=42,
    shuffle=True
)

In [112]:
val_reports, test_reports = train_test_split(
    temp_reports,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

In [113]:
train_df = df_full[df_full["report"].isin(train_reports)]
val_df   = df_full[df_full["report"].isin(val_reports)]
test_df  = df_full[df_full["report"].isin(test_reports)]

In [119]:
len(train_reports), len(val_reports), len(test_reports)

(37, 13, 13)

In [120]:
len(train_df), len(val_df), len(test_df)

(903, 301, 301)

In [121]:
train_df = train_df.dropna()
val_df = val_df.dropna()
test_df = test_df.dropna()

In [122]:
len(train_df), len(val_df), len(test_df)

(869, 291, 292)

In [123]:
train_df.to_csv("/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/training_data/company_description_2" + "/train_data.csv", index=False)
val_df.to_csv("/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/training_data/company_description_2" + "/val_data.csv", index=False)
test_df.to_csv("/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/training_data/company_description_2" + "/test_data.csv", index=False)